# 🌀 The Kernel Trick in SVM

<a href="https://colab.research.google.com/github/rubenfonnegra/machine_learning/blob/master/Sem_04/svm_kernel_trick.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> 
<a href="https://github.com/rubenfonnegra/machine_learning/blob/master/Sem_04/svm_kernel_trick.ipynb" target="_parent"><img src="https://img.shields.io/badge/%E2%80%8B-Open%20in%20Github-blue?logo=github" alt="Open In Github"/></a> 


### Learning objectives

By the end of this notebook, you will be able to:

- Explain why linear SVMs fail on nonlinear data.
- Understand explicit feature mappings and the kernel trick.
- Visualize concentric 2D data and map it into 3D.
- Train linear, polynomial, RBF, and sigmoid SVMs.
- Compare hyperparameters and decision boundaries.
- Apply different kernels to a real dataset.


### Documentation

- [```make_blobs``` ](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_blobs.html#sklearn.datasets.make_blobs)
- [```SVC``` ](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html#sklearn.svm.SVC)

---

> **📘 Machine Learning**  
> **Author:** Rubén D. Fonnegra, Ph.D. \
> **Institution:** Institución Universitaria Pascual Bravo  
> © 2026 · Educational use with attribution

### Imports

In [ ]:
#@markdown #### **🛠️⚙️📦 Install complementary dependencies**. 

from tqdm.auto import tqdm
import subprocess, time, sys

LIB = "MLTools-1.2-py3-none-any.whl"
URL = "https://drive.google.com/uc?id=18Y834Tvtj_-Px9yNmbAB4yV4L0gcIZ20"

commands = [
    ("📦 Downloading resources", ["gdown", URL, "-O", LIB], 35),
    ("🔧 Installing dependencies", [sys.executable, "-m", "pip", "install", "-q", LIB], 55),
    ("🧹 Finishing", ["rm", "-f", LIB], 10)
]

print("⚙️ Iniciando configuración del entorno...\n")

try:
    with tqdm(total=100, desc="Preparando", bar_format="{desc}: {bar} {n:.0f}%") as bar:
        for label, command, weight in commands:
            bar.set_description(label)
            subprocess.run(command, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
            for _ in range(weight):
                time.sleep(0.01)
                bar.update(1)

    print("\n✅ Configuración completada correctamente. Puede comenzar la actividad.")

except subprocess.CalledProcessError as e:
    print("\n❌ Error durante la configuración")
    print(f"Exit code: {e.returncode}")

    if e.stdout:
        print("\n📤 STDOUT:")
        print(e.stdout)

    if e.stderr:
        print("\n🔍 STDERR:")
        print(e.stderr)

    print("\n❌ No fue posible configurar el entorno. Ejecute nuevamente la celda.")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_circles
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

from MLTools import generate_isotropic_circles, plot_decision_boundaries

### Linear SVM Review

A linear SVM uses

$$f(\mathbf{x})=\mathbf{m}^{T}\mathbf{x}+b$$

and creates a straight boundary in 2D. Nonlinear data may require a feature transformation.


### Isotropic Concentric Data in 2D

In [ ]:
X, y = generate_isotropic_circles(n_samples = _ , random_state = 10)

plt.figure(figsize=(8, 7))

plt.scatter(X[:, 0], X[:, 1], c = y, alpha=0.75, cmap = 'Paired')
plt.xlabel("x1")
plt.ylabel("x2")
plt.title("Concentric Isotropic Data")
plt.axis("equal")
plt.legend()
plt.show()


### Linear SVM on the Original Data

In [ ]:
linear_model = SVC(kernel="linear", C=1.0)
linear_model.fit( _ , _ )
pred = linear_model.predict( _ )

print("Accuracy:", accuracy_score( _ , _ ))

### Decision-Boundary Function

In [ ]:
plot_decision_boundaries( _ , _ , _ , resolution = 100, padding=0.1, 
                         feature_names = ["x1", "x2"], title = "Linear SVM on Concentric Data")

### Explicit Mapping into 3D

Create a radial feature:

$$
z=x_1^2+x_2^2
$$

The two circular classes now occupy different vertical regions.


In [ ]:
x1 = X[:, 0]
x2 = X[:, 1]
z = x1**2 + x2**2

X3 = np.column_stack([x1, x2, z])

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")
ax.view_init(90,270)

for label in [0, 1]:
    mask = y == label
    ax.scatter(
        X3[mask, 0],
        X3[mask, 1],
        X3[mask, 2],
        label=f"Class {label}",
        alpha=0.7
    )

ax.set_xlabel("x1")
ax.set_ylabel("x2")
ax.set_zlabel("z = x1² + x2²")
ax.set_title("Mapped Data in 3D")
ax.legend()
plt.show()


### Linear SVM in the 3D Space

In [ ]:
linear_3d = SVC(kernel="linear", C=1.0)

linear_3d.fit( _ , _ )
pred3 = linear_3d.predict( _ )

print("3D linear SVM accuracy:", accuracy_score( _ , _ ))


### Separating Plane in 3D

Step 1. Start from the equation of the separating hyperplane

$$w_0x + w_1y + w_2z + b = 0$$

Step 2. Solve the equation for $z$. First, move the remaining terms to the right-hand side:

$$
w_2z = -\left(w_0x + w_1y + b\right)
$$

Then divide both sides by w3:

$$
z = -\frac{w_0x + w_1y + b}{w_2}
$$

Then:

$$
z = -\frac{w_1x + w_2y + b}{w_3}
\quad\Longrightarrow\quad
\texttt{gz = -(w[0] * gx + w[1] * gy + b) / w[2]}
$$

In [ ]:
w = linear_3d.coef_[0]
b = linear_3d.intercept_[0]

gx, gy = np.meshgrid(
    np.linspace(X3[:, 0].min(), X3[:, 0].max(), 30),
    np.linspace(X3[:, 1].min(), X3[:, 1].max(), 30)
)

gz = -(w[0] * gx + w[1] * gy + b) / w[2]

fig = plt.figure(figsize=(11, 8))
ax = fig.add_subplot(111, projection="3d")
ax.view_init(90,270)

for label in [0, 1]:
    mask = y == label
    ax.scatter(X3[mask, 0], X3[mask, 1], X3[mask, 2], label=f"Class {label}", alpha=0.6)

ax.plot_surface(gx, gy, gz, alpha=0.25)
ax.set_xlabel("x1")
ax.set_ylabel("x2")
ax.set_zlabel("z")
ax.set_title("Linear Separation in the 3D Feature Space")
ax.legend()
plt.show()


### The Kernel Trick

A kernel computes inner products in a transformed feature space without explicitly creating all transformed coordinates:

$$
K(\mathbf{x},\mathbf{y})=\phi(\mathbf{x})^T\phi(\mathbf{y})
$$

Common kernels include:

- Linear
- Polynomial
- Radial Basis Function (RBF)
- Sigmoid


### Compare Kernels on the Circular Data

In [ ]:

kernels = ['linear', 'poly', 'rbf']

_, axes = plt.subplots(1,3, figsize = (15,5))

# fit the model
for kernel, ax in zip( _ , _ ):
    #
    clf = SVC(kernel = _ , gamma = 1)
    clf.fit( _ , _ )
    plot_decision_boundaries( _ , _ , _ , resolution = 100, padding=0.1, ax = ax, 
                                  feature_names = ["x1", "x2"], title = f'Kernel= {kernel}')
    
    
plt.show()

### Effect of gamma in the RBF Kernel

In [ ]:

gammas = [0.1, 1, 10]

_, axes = plt.subplots(1,3, figsize = (15,5))

# fit the model
for ax, gamma in zip( _ , _ ):
    #
    clf = SVC(kernel = 'rbf' , gamma = _ )
    clf.fit( _ , _ )
    plot_decision_boundaries( _ , _ , _ , resolution = 100, padding=0.1, ax = ax, 
                              feature_names = ["x1", "x2"], title = f'Gamma= {gamma}')
    
    
plt.show()

### Interpretation

- **Linear:** straight boundaries and lower complexity.
- **Polynomial:** curved algebraic boundaries.
- **RBF:** flexible local boundaries.
- **Sigmoid:** neural-network-like shape, but often sensitive to tuning.

More complex boundaries do not automatically generalize better.


### Practice Exercises

#### Exercise 1
Compare a linear SVM in 2D with a linear SVM after adding `z=x1**2+x2**2`.


In [ ]:
# Your code here

#### Exercise 2
Test RBF kernels with several values of `gamma` and `C`.


In [ ]:
# Your code here

#### Exercise 3
Compare polynomial degrees 2, 3, 4, and 5.


In [ ]:
# Your code here

#### Exercise 4
Choose the best kernel for the Iris example and justify your answer using accuracy, macro F1, support vectors, and boundary complexity.


In [ ]:
# Your explanation here

#### Final Challenge ⭐

Build a complete kernel-SVM comparison:

1. Load a classification dataset.
2. Select two features.
3. Split and scale the data.
4. Train linear, polynomial, RBF, and sigmoid SVMs.
5. Tune at least one hyperparameter per nonlinear kernel.
6. Compare accuracy, macro F1, and support-vector count.
7. Plot all decision boundaries.
8. Identify underfitting or overfitting.
9. Recommend one kernel.


In [ ]:
# Your code here

---

## 📄 Attribution

This notebook was developed by **Rubén D. Fonnegra** as educational material for Machine Learning courses at **Institución Universitaria Pascual Bravo**.
You may use, share, and adapt this material for educational purposes, provided that appropriate credit is given to the original author.

**Suggested citation:**
> Fonnegra Tarazona, R. D. (2026). *The Kernel Trick in SVM: Machine Learning Notebook*. Institución Universitaria Pascual Bravo.